In [28]:
import pandas as pd
import numpy as np
import datetime
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm

import joblib
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
#from 04_feature_engineering import FeaturePipeline

In [29]:
FEATURES = [
    'Store_id',#'Store_Type','Location_Type','Region_Code',
    'Store_Type_S1',
       'Store_Type_S2', 'Store_Type_S3', 'Store_Type_S4', 'Location_Type_L1',
       'Location_Type_L2', 'Location_Type_L3', 'Location_Type_L4',
       'Location_Type_L5', 'Region_Code_R1', 'Region_Code_R2',
       'Region_Code_R3', 'Region_Code_R4',
    'Discount_Flag','Holiday',
    'day','month','weekday','weekofyear','is_weekend',
    'lag_1_#Order','lag_7_#Order','rolling_#Order_7',
    'lag_1_Sales','lag_7_Sales','rolling_Sales_7',
    'lag_1_aov',
    'Sales','#Order'
]
FEATURES_ORDERS = [
    'Store_id',#'Store_Type','Location_Type','Region_Code',
    'Store_Type_S1',
       'Store_Type_S2', 'Store_Type_S3', 'Store_Type_S4', 'Location_Type_L1',
       'Location_Type_L2', 'Location_Type_L3', 'Location_Type_L4',
       'Location_Type_L5', 'Region_Code_R1', 'Region_Code_R2',
       'Region_Code_R3', 'Region_Code_R4',
    'Discount_Flag','Holiday',
    'day','month','weekday','weekofyear','is_weekend',
    'lag_1_#Order',
    'lag_7_#Order','rolling_#Order_7',
    'lag_1_Sales',
    'lag_7_Sales','rolling_Sales_7',
    'lag_1_aov'
]

FEATURES_SALES = [
    'Store_id',#'Store_Type','Location_Type','Region_Code',
    'Store_Type_S1',
       'Store_Type_S2', 'Store_Type_S3', 'Store_Type_S4', 'Location_Type_L1',
       'Location_Type_L2', 'Location_Type_L3', 'Location_Type_L4',
       'Location_Type_L5', 'Region_Code_R1', 'Region_Code_R2',
       'Region_Code_R3', 'Region_Code_R4',
    'Discount_Flag','Holiday',
    'day','month','weekday','weekofyear','is_weekend',
    'lag_1_#Order',
    'lag_7_#Order','rolling_#Order_7',
    'lag_1_Sales',
    'lag_7_Sales','rolling_Sales_7',
    'lag_1_aov','pred_orders'
]

column_dtype_map_dict = {'Store_id': 'int64',
 'Store_Type_S1': 'int64',
 'Store_Type_S2': 'int64',
 'Store_Type_S3': 'int64',
 'Store_Type_S4': 'int64',
 'Location_Type_L1': 'int64',
 'Location_Type_L2': 'int64',
 'Location_Type_L3': 'int64',
 'Location_Type_L4': 'int64',
 'Location_Type_L5': 'int64',
 'Region_Code_R1': 'int64',
 'Region_Code_R2': 'int64',
 'Region_Code_R3': 'int64',
 'Region_Code_R4': 'int64',
 'Discount_Flag': 'int64',
 'Holiday': 'int64',
 'day': 'int64',
 'month': 'int64',
 'weekday': 'int64',
 'weekofyear': 'int64',
 'is_weekend': 'int64',
 'lag_1_#Order': 'float64',
 'lag_7_#Order': 'float64',
 'rolling_#Order_7': 'float64',
 'lag_1_Sales': 'float64',
 'lag_7_Sales': 'float64',
 'rolling_Sales_7': 'float64',
 'lag_1_aov': 'float64',
 'Sales': 'float64',
 '#Order': 'float64'}

In [30]:
mlflow.set_experiment("Retail_Forecasting")

<Experiment: artifact_location=('file:d:/Study/Scaler/DS and ML/Recordings and Notes/Final Year project '
 'files/Final '
 'Year/code/product-sales-forecasting-end-to-end/notebooks/mlruns/1'), creation_time=1776596086867, experiment_id='1', last_update_time=1776596086867, lifecycle_stage='active', name='Retail_Forecasting', tags={}, workspace='default'>

In [31]:
def prepare_data(train_df, val_df):
    #pipeline = FeaturePipeline()

    #train_df = pipeline.fit_transform(train_df)
    #val_df = pipeline.transform(val_df)
    
    train_df = train_df.dropna()
    val_df = val_df.dropna()

    X_train = train_df[FEATURES]
    X_val = val_df[FEATURES]


    y_train_sales = train_df['Sales']
    y_val_sales = val_df['Sales']

    y_train_orders = train_df['#Order']
    y_val_orders = val_df['#Order']

    return (
        X_train, X_val,
        y_train_sales, y_val_sales,
        y_train_orders, y_val_orders,
        True
        #pipeline
    )

In [32]:
def log_metrics(y_val_sales, pred_sales, y_val_orders, pred_orders):
    sales_mae = mean_absolute_error(y_val_sales, pred_sales)
    orders_mae = mean_absolute_error(y_val_orders, pred_orders)

    mlflow.log_metric("sales_mae", sales_mae)
    mlflow.log_metric("orders_mae", orders_mae)

    print("Sales MAE:", sales_mae)
    print("Orders MAE:", orders_mae)

In [39]:
def train_linear_model(train_df, val_df):
    start_timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S") 
    with mlflow.start_run(run_name=f"Linear_Model_Ridge_{start_timestamp}"):

        (
            X_train, X_val,
            y_train_sales, y_val_sales,
            y_train_orders, y_val_orders,
            pipeline
        ) = prepare_data(train_df, val_df)

        model_sales = Ridge(alpha=49.96758009381003)
        model_orders = Ridge(alpha=49.927017883207526)

        # log params
        mlflow.log_param("model", "Ridge")
        mlflow.log_param("alpha", 1.0)

        model_orders.fit(X_train[FEATURES_ORDERS], y_train_orders)

        X_train['pred_orders'] = model_orders.predict(X_train[FEATURES_ORDERS])

        model_sales.fit(X_train[FEATURES_SALES], y_train_sales)
        
        pred_orders = model_orders.predict(X_val[FEATURES_ORDERS])
        X_val['pred_orders'] = pred_orders
        pred_sales = model_sales.predict(X_val[FEATURES_SALES])


        log_metrics(y_val_sales, pred_sales, y_val_orders, pred_orders)

        # log models
        mlflow.sklearn.log_model(model_sales, "linear_sales_model")
        mlflow.sklearn.log_model(model_orders, "linear_orders_model")
        #mlflow.sklearn.log_model(pipeline, "pipeline")

In [34]:
def train_xgboost_model(train_df, val_df):
    start_timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S") 
    with mlflow.start_run(run_name=f"XGBoost_Model_{start_timestamp}"):

        (
            X_train, X_val,
            y_train_sales, y_val_sales,
            y_train_orders, y_val_orders,
            pipeline
        ) = prepare_data(train_df, val_df)

        #params = {
        #     "n_estimators": 500,
        #     "learning_rate": 0.05,
        #     "max_depth": 6,
        #     "subsample": 0.8,
        #     "colsample_bytree": 0.8,
        #     "random_state": 42
        # }
        order_params = {
        'n_estimators': 201,
        'max_depth': 10,
        'learning_rate': 0.013960663099383598,
        'subsample': 0.9208699091164867,
        'colsample_bytree': 0.9170140235850499,
        'reg_alpha': 4.824825650018285,
        'reg_lambda': 2.4528319109319674
        }

        sales_params = {
            'n_estimators': 323,
            'max_depth': 5,
            'learning_rate': 0.026530195738347556,
            'subsample': 0.7388544536271039,
            'colsample_bytree': 0.8628252390092686,
            'reg_alpha': 1.1499809816719329,
            'reg_lambda': 4.984633296547041
        }

        model_orders = XGBRegressor(**order_params)
        model_sales = XGBRegressor(**sales_params)
        

        mlflow.log_param("model", "XGBoost")
        #mlflow.log_param("parameters_for", "Orders")
        #mlflow.log_params(order_params)
        #mlflow.log_param("parameters_for", "Sales")
        #mlflow.log_params(sales_params)

        model_orders.fit(X_train[FEATURES_ORDERS], y_train_orders)

        X_train['pred_orders'] = model_orders.predict(X_train[FEATURES_ORDERS])

        model_sales.fit(X_train[FEATURES_SALES], y_train_sales)
        
        pred_orders = model_orders.predict(X_val[FEATURES_ORDERS])
        X_val['pred_orders'] = pred_orders
        pred_sales = model_sales.predict(X_val[FEATURES_SALES])


        log_metrics(y_val_sales, pred_sales, y_val_orders, pred_orders)

        mlflow.xgboost.log_model(model_sales, "xgb_sales_model")
        mlflow.xgboost.log_model(model_orders, "xgb_orders_model")
        #mlflow.sklearn.log_model(pipeline, "pipeline")

In [35]:
def train_lightgbm_model(train_df, val_df):
    start_timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S") 
    with mlflow.start_run(run_name=f"LightGBM_Model_{start_timestamp}"):

        (
            X_train, X_val,
            y_train_sales, y_val_sales,
            y_train_orders, y_val_orders,
            pipeline
        ) = prepare_data(train_df, val_df)

        # params = {
        #     "n_estimators": 500,
        #     "learning_rate": 0.05,
        #     "num_leaves": 31,
        #     "subsample": 0.8,
        #     "colsample_bytree": 0.8,
        #     "random_state": 42
        # }

        order_params = {
            'n_estimators': 343,
            'num_leaves': 114,
            'learning_rate': 0.01108019477729205,
            'subsample': 0.6017693529578273,
            'colsample_bytree': 0.7481295732862226
            }
        
        sales_params = {
            'n_estimators': 294,
            'num_leaves': 113,
            'learning_rate': 0.014560582047605566,
            'subsample': 0.8220865020527265,
            'colsample_bytree': 0.9632820171720197
        }

        model_sales = LGBMRegressor(**sales_params)
        model_orders = LGBMRegressor(**order_params)

        mlflow.log_param("model", "LightGBM")
        #mlflow.log_param("parameters_for", "Orders")
        #mlflow.log_params(order_params)
        #mlflow.log_param("parameters_for", "Sales")
        #mlflow.log_params(sales_params)

        model_orders.fit(X_train[FEATURES_ORDERS], y_train_orders)

        X_train['pred_orders'] = model_orders.predict(X_train[FEATURES_ORDERS])

        model_sales.fit(X_train[FEATURES_SALES], y_train_sales)
        
        pred_orders = model_orders.predict(X_val[FEATURES_ORDERS])
        X_val['pred_orders'] = pred_orders
        pred_sales = model_sales.predict(X_val[FEATURES_SALES])
        

        log_metrics(y_val_sales, pred_sales, y_val_orders, pred_orders)

        mlflow.lightgbm.log_model(model_sales, "lgbm_sales_model")
        mlflow.lightgbm.log_model(model_orders, "lgbm_orders_model")
        #mlflow.sklearn.log_model(pipeline, "pipeline")

In [ ]:
# Load the datasets
train_df = pd.read_csv('../data/experiments/train/train_transformed_data.csv')
val_df = pd.read_csv('../data/experiments/validate/validate_transformed_data.csv')

In [37]:
def format_df_dtype(df, features_dtype_map_dict):
    for col, dtype in features_dtype_map_dict.items():
        df[col] = df[col].astype(dtype)
    return df

train_df = format_df_dtype(train_df, column_dtype_map_dict)
val_df = format_df_dtype(val_df, column_dtype_map_dict)

In [38]:
train_lightgbm_model(train_df, val_df)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004521 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2177
[LightGBM] [Info] Number of data points in the train set: 141985, number of used features: 28
[LightGBM] [Info] Start training from score 67.836138
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009714 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2432
[LightGBM] [Info] Number of data points in the train set: 141985, number of used features: 29
[LightGBM] [Info] Start training from score 42618.702030


2026/04/21 01:16:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/21 01:16:34 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Sales MAE: 6473.314014894356
Orders MAE: 10.351978542931649


2026/04/21 01:16:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/21 01:16:43 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [24]:
train_xgboost_model(train_df, val_df)

2026/04/21 00:23:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Sales MAE: 6359.33761533226
Orders MAE: 10.236441360139704


2026/04/21 00:23:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [40]:
train_df2=train_df.copy(deep=True).dropna()
val_df2=val_df.copy(deep=True).dropna()

train_linear_model(train_df2, val_df2)

2026/04/21 01:36:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/21 01:36:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Sales MAE: 6918.373813521993
Orders MAE: 11.002579358863068


2026/04/21 01:37:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/21 01:37:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
